# 3 - Python + Stata

`%%stata` is **PyStata's own magic** - EconEnv loads it rather than
reimplementing it. What EconEnv adds is discovery, lifecycle, structured
results and full-precision data access.

In [ ]:
%load_ext econenv

In [ ]:
%econ engines

In [ ]:
import numpy as np
import pandas as pd
import econenv

rng = np.random.default_rng(20260904)
n = 120
df = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.normal(size=n)})
df['y'] = 2.0 + 0.5 * df.x1 - 0.3 * df.x2 + rng.normal(scale=0.4, size=n)
df.head()

## Load and estimate

In [ ]:
%stata_push df

In [ ]:
%%stata
summarize
regress y x1 x2

## Results at full precision

`display` rounds to about nine significant figures. These go through Stata's
Function Interface, so they are the stored doubles.

In [ ]:
stata = econenv.engine('stata')
print('e(r2) =', repr(stata.pull_scalar('e(r2)')))
print('e(N)  =', stata.pull_scalar('e(N)'))

In [ ]:
table = stata.pull_matrix('r(table)')
print(table.shape)
table[:6]

## Frames

In [ ]:
econenv.push('stata', 'subsample', df.head(50))

In [ ]:
%%stata
frame change subsample
count
frame change default

## Errors are translated, not swallowed

In [ ]:
try:
    stata.execute('regress y nosuchvar')
except Exception as exc:
    print(exc)